<a href="https://colab.research.google.com/github/moujar/PathOptLearn/blob/Mistral-Trials/adaptive_learning_finetune_pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adaptive Video Learning System - SciQ Fine-tuning (Optimized for Pro)
Fine-tuning Mistral 7B on SciQ dataset - Optimized for A100 GPU

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q datasets transformers accelerate peft bitsandbytes trl torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 45.0 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer
import pandas as pd
import random

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.47 GB


In [ ]:
# Check what GPU you actually got
!nvidia-smi

# Check if you're actually using the A100
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"Memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

Sat Jan 24 15:22:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             47W /  400W |       5MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Load and Explore SciQ Dataset

In [ ]:
# Load SciQ dataset from HuggingFace
dataset = load_dataset("allenai/sciq")

print("Dataset structure:")
print(dataset)
print("\nSample from training set:")
print(dataset['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.99M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/339k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/343k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11679 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 11679
    })
    validation: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 1000
    })
})

Sample from training set:
{'question': 'What type of organism is commonly used in preparation of foods such as cheese and yogurt?', 'distractor3': 'viruses', 'distractor1': 'protozoa', 'distractor2': 'gymnosperms', 'correct_answer': 'mesophilic organisms', 'support': 'Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many p

In [ ]:
# Explore the dataset structure
sample = dataset['train'][0]
print("Dataset fields:")
for key, value in sample.items():
    print(f"\n{key}: {value}")

Dataset fields:

question: What type of organism is commonly used in preparation of foods such as cheese and yogurt?

distractor3: viruses

distractor1: protozoa

distractor2: gymnosperms

correct_answer: mesophilic organisms

support: Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.


In [ ]:
# Check dataset sizes
print(f"Training set size: {len(dataset['train'])}")
print(f"Validation set size: {len(dataset['validation'])}")
print(f"Test set size: {len(dataset['test'])}")

Training set size: 11679
Validation set size: 1000
Test set size: 1000


## 3. Prepare Dataset for Fine-tuning

In [ ]:
def create_training_prompt(example):
    """
    Format: We'll create examples where the model needs to:
    1. Evaluate if student understood the concept
    2. Recommend next steps

    For training, we'll use the correct answer as the "correct student response"
    and distractors as "incorrect student responses"
    """

    context = example['support']
    question = example['question']
    correct_answer = example['correct_answer']

    # For correct answer scenario
    correct_prompt = f"""### Context (Video Transcript):
{context}

### Assessment Question:
{question}

### Student Answer:
{correct_answer}

### Evaluation:
The student has demonstrated strong understanding of the concept. Their answer correctly identifies {correct_answer}, showing they've grasped the key principles from the lesson.

### Recommendation:
The student is ready to progress to more advanced topics building on this foundation. Recommend moving forward to the next concept in the learning sequence."""

    return correct_prompt

def create_incorrect_prompt(example):
    """Create prompts for incorrect answer scenarios using distractors"""
    context = example['support']
    question = example['question']
    correct_answer = example['correct_answer']

    # Pick a random distractor
    distractors = [example['distractor1'], example['distractor2'], example['distractor3']]
    incorrect_answer = random.choice([d for d in distractors if d])  # Filter out empty distractors

    if not incorrect_answer:
        return None

    incorrect_prompt = f"""### Context (Video Transcript):
{context}

### Assessment Question:
{question}

### Student Answer:
{incorrect_answer}

### Evaluation:
The student's answer shows a misconception. They answered '{incorrect_answer}' but the correct answer is '{correct_answer}'. This indicates they need to review the foundational concepts.

### Recommendation:
The student should review prerequisite material before proceeding. Recommend watching an introductory video on this topic or revisiting related foundational concepts."""

    return incorrect_prompt

# Test the prompt formatting
print("SAMPLE CORRECT ANSWER PROMPT:")
print("=" * 80)
print(create_training_prompt(dataset['train'][0]))
print("\n" + "=" * 80)
print("\nSAMPLE INCORRECT ANSWER PROMPT:")
print("=" * 80)
print(create_incorrect_prompt(dataset['train'][0]))

SAMPLE CORRECT ANSWER PROMPT:
### Context (Video Transcript):
Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.

### Assessment Question:
What type of organism is commonly used in preparation of foods such as cheese and yogurt?

### Student Answer:
mesophilic organisms

### Evaluation:
The student has demonstrated strong understanding of the concept. Their answer correctly identifies mesophilic organisms, showing they've grasped the key principles from the lesson.

### Recommendation:
The student is ready to progress to more advanced topics building on this foundation. Recommend moving forward to the next concept in the learning sequence.


SAMPLE I

In [ ]:
# Create balanced dataset with both correct and incorrect examples
def prepare_dataset(dataset_split):
    prompts = []

    for example in dataset_split:
        # Add correct answer example
        prompts.append(create_training_prompt(example))

        # Add incorrect answer example
        incorrect = create_incorrect_prompt(example)
        if incorrect:
            prompts.append(incorrect)

    return prompts

# Prepare training data
train_prompts = prepare_dataset(dataset['train'])
val_prompts = prepare_dataset(dataset['validation'])

print(f"Total training examples: {len(train_prompts)}")
print(f"Total validation examples: {len(val_prompts)}")

# Create dataset dict for training
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": train_prompts})
val_dataset = Dataset.from_dict({"text": val_prompts})

print("\nDataset prepared and ready for fine-tuning!")

Total training examples: 23358
Total validation examples: 2000

Dataset prepared and ready for fine-tuning!


## 4. Load Mistral 7B Model with QLoRA

In [ ]:
# Model configuration
model_name = "mistralai/Mistral-7B-v0.1"

# Quantization config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model and tokenizer loaded successfully!")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Model and tokenizer loaded successfully!


## 5. Configure LoRA for Efficient Fine-tuning

In [ ]:
# LoRA configuration - Optimized for A100
peft_config = LoraConfig(
    r=16,  # LoRA rank
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# Prepare model for training
model = prepare_model_for_kbit_training(model)

print("Model prepared for training!")

Model prepared for training!


## 6. Training Configuration (Optimized for A100)

In [ ]:
# Training arguments - OPTIMIZED FOR SPEED
training_args = TrainingArguments(
    output_dir="./adaptive-learning-mistral",
    num_train_epochs=2,  # Reduced from 3 to 2 epochs
    per_device_train_batch_size=8,  # Increased from 4 to 8 - A100 can handle this
    per_device_eval_batch_size=8,  # Increased from 4 to 8
    gradient_accumulation_steps=2,  # Reduced from 4 to 2 (effective batch size still 16)
    learning_rate=2e-4,
    bf16=True,
    save_strategy="steps",
    save_steps=200,  # Save less frequently (was 100)
    eval_strategy="steps",
    eval_steps=200,  # Eval less frequently (was 100)
    logging_steps=20,  # Log less frequently (was 10)
    warmup_steps=100,  # Increased warmup slightly
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    save_total_limit=2,
    load_best_model_at_end=True,
    dataloader_num_workers=2,  # Added parallel data loading
    gradient_checkpointing=True,  # Enable gradient checkpointing for memory
    max_grad_norm=0.3,  # Add gradient clipping for stability
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    args=training_args,
)

print("Trainer initialized and ready!")
print(f"\nTraining Configuration:")
print(f"  - Epochs: {training_args.num_train_epochs}")
print(f"  - Batch size per device: {training_args.per_device_train_batch_size}")
print(f"  - Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"  - Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  - Total training steps: ~{len(train_dataset) * training_args.num_train_epochs // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/23358 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/23358 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/23358 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Trainer initialized and ready!

Training Configuration:
  - Epochs: 2
  - Batch size per device: 8
  - Gradient accumulation steps: 2
  - Effective batch size: 16
  - Total training steps: ~2919


## 7. Start Training

In [ ]:
# Start training
print("Starting training...")
print("This should take approximately 1-2 hours on A100")
trainer.train()

print("\nTraining completed!")

Starting training...
This should take approximately 1-2 hours on A100


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
200,0.746700,0.724578


wandb: WARNING URL not available in offline run


Step,Training Loss,Validation Loss
200,0.746700,0.724578
400,0.735500,0.721848
600,0.715100,0.717448
800,0.720700,0.717120
1000,0.691000,0.716430
1200,0.667600,0.716853
1400,0.637500,0.717636
1600,0.498600,0.744434
1800,0.502500,0.749240
2000,0.496300,0.752413


wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run



Training completed!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 8. Save the Fine-tuned Model

In [ ]:
# Save the fine-tuned model
trainer.model.save_pretrained("./adaptive-learning-mistral-final")
tokenizer.save_pretrained("./adaptive-learning-mistral-final")

print("Model saved successfully!")

Model saved successfully!


## 9. Test the Model

In [ ]:
# Test with a sample from the test set
test_example = dataset['test'][0]

# Create test prompt with student answer
test_prompt = f"""### Context (Video Transcript):
{test_example['support']}

### Assessment Question:
{test_example['question']}

### Student Answer:
{test_example['correct_answer']}

### Evaluation:"""

# Generate response
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = trainer.model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    top_p=0.95,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("MODEL RESPONSE (CORRECT ANSWER):")
print("=" * 80)
print(response)
print("=" * 80)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


MODEL RESPONSE (CORRECT ANSWER):
### Context (Video Transcript):
Oxidants and Reductants Compounds that are capable of accepting electrons, such as O 2 or F2, are calledoxidants (or oxidizing agents) because they can oxidize other compounds. In the process of accepting electrons, an oxidant is reduced. Compounds that are capable of donating electrons, such as sodium metal or cyclohexane (C6H12), are calledreductants (or reducing agents) because they can cause the reduction of another compound. In the process of donating electrons, a reductant is oxidized. These relationships are summarized in Equation 3.30: Equation 3.30 Saylor URL: http://www. saylor. org/books.

### Assessment Question:
Compounds that are capable of accepting electrons, such as o 2 or f2, are called what?

### Student Answer:
oxidants

### Evaluation:
The student has demonstrated strong understanding of the concept. Their answer correctly identifies oxidants, showing they've grasped the key principles from the lesson

In [ ]:
# Test with incorrect answer
test_prompt_incorrect = f"""### Context (Video Transcript):
{test_example['support']}

### Assessment Question:
{test_example['question']}

### Student Answer:
{test_example['distractor1']}

### Evaluation:"""

inputs = tokenizer(test_prompt_incorrect, return_tensors="pt").to("cuda")
outputs = trainer.model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    top_p=0.95,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("MODEL RESPONSE (INCORRECT ANSWER):")
print("=" * 80)
print(response)
print("=" * 80)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


MODEL RESPONSE (INCORRECT ANSWER):
### Context (Video Transcript):
Oxidants and Reductants Compounds that are capable of accepting electrons, such as O 2 or F2, are calledoxidants (or oxidizing agents) because they can oxidize other compounds. In the process of accepting electrons, an oxidant is reduced. Compounds that are capable of donating electrons, such as sodium metal or cyclohexane (C6H12), are calledreductants (or reducing agents) because they can cause the reduction of another compound. In the process of donating electrons, a reductant is oxidized. These relationships are summarized in Equation 3.30: Equation 3.30 Saylor URL: http://www. saylor. org/books.

### Assessment Question:
Compounds that are capable of accepting electrons, such as o 2 or f2, are called what?

### Student Answer:
antioxidants

### Evaluation:
The student's answer shows a misconception. They answered 'antioxidants' but the correct answer is 'oxidants'. This indicates they need to review the foundational

## 10. Save to Google Drive (Optional)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy model to Drive
!cp -r ./adaptive-learning-mistral-final /content/drive/MyDrive/

print("Model saved to Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model saved to Google Drive!
